# DNAscape mapping atlas

This notebook is organised by **target feature**. The guiding question is: _what replication feature do I want to obtain, and what data or assumptions do I need to get it?_

DNAscape maps between replication-timing profiles, origin-firing or initiation programmes, fork-speed descriptions, and richer replication fields such as the replicated fraction `f(x,t)`. The notebook focuses first on the central kinetic triangle of the software:

1. **Replication timing**, usually compared with Repli-seq or related timing profiles.
2. **Origin firing / initiation rate**, compared with origin-associated data such as SNS-seq, Ini-seq, ORC/MCM occupancy, chromatin accessibility, or inferred initiation tracks.
3. **Fork speed**, relevant to DNA fibre/combing, Fork-seq, polymerase progression, and replication-stress perturbations.

DNAscape helps users combine these data types, test whether different assays are mutually consistent under a shared kinetic interpretation, and ask which replication feature must change to explain a measured phenotype. The arrays used by the model are kinetic interpretations rather than raw assay values; they should be normalised, aligned, smoothed, resampled, and otherwise preprocessed with the biological question in mind. See `1_datamanagement.ipynb` for data-management details.

## Setup

Autoreload is useful while DNAscape is under active development. Re-run this cell after editing the source code.


In [ ]:
%reload_ext autoreload
%autoreload 2

from dnascape import *

## Example data

We use the bundled hESC chromosome 1 replication-timing profile. The loaded timing profile is a one-dimensional array over genomic bins, and `positions_kb` stores the corresponding genomic coordinate in kilobases. The same structure applies to any regularly binned timing track.


In [ ]:
chr_number = 1
output_resolution_kb = 1
interval_kb = (0, 60_000)
fork_speed = 1.4

replication_timing_full = loadcsv(
    "example",
    example="replication_timing_hESC",
    chr_number=chr_number,
)

replication_timing, positions_kb = select_profile_region(
    replication_timing_full,
    output_resolution_kb=output_resolution_kb,
    interval_kb=interval_kb,
)

In [ ]:
plotf(
    replication_timing,
    x_array=positions_kb,
    invyQ=True,
    xtitle="Chromosome position (kb)",
    ytitle="Replication timing (min)",
)

## Core mapping routes

For the core DNAscape mapping problem, choose the target feature first and then ask which other kinetic descriptions are supplied as data or modelling inputs. This is the practical triangulation used throughout the software: replication timing, origin firing, and fork speed constrain one another.

The table below lays out the full route space for the three central quantities. Replication timing is represented as `T(x)`. Initiation and fork speed are listed across four forms: constant (`I0`, `v0`), spatial (`I(x)`, `v(x)`), time-dependent but spatially uniform (`I(t)`, `v(t)`), and fully space-time dependent (`I(x,t)`, `v(x,t)`). In code, these correspond naturally to a scalar, a one-dimensional array, a row-vector-like two-dimensional array such as `[[...]]`, and a two-dimensional matrix.

The column corresponding to the target is shown as `-`, because that quantity is being inferred rather than supplied. Rows marked **implemented** are available now; rows marked **needs method** are valid routes to evaluate and implement separately. The broader object behind these summaries is the replicated fraction field `f(x,t)`, while replication timing is often the most data-relatable one-dimensional summary of that field.

In [ ]:
I_FORMS = ["I0", "I(x)", "I(t)", "I(x,t)"]
V_FORMS = ["v0", "v(x)", "v(t)", "v(x,t)"]

rows = []

for firing_form in I_FORMS:
    for fork_speed_form in V_FORMS:
        implemented = firing_form == "I(x)" and fork_speed_form == "v0"
        rows.append({
            "target": "replication_timing",
            "target form": "T(x)",
            "required data": "firing_rate, fork_speed",
            "timing data": "-",
            "firing data": firing_form,
            "fork-speed data": fork_speed_form,
            "method": "firing_to_timing_deterministic" if implemented else "-",
            "status": "implemented" if implemented else "needs method",
        })

for target_form in I_FORMS:
    for fork_speed_form in V_FORMS:
        implemented = target_form == "I(x)" and fork_speed_form == "v0"
        rows.append({
            "target": "firing_rate",
            "target form": target_form,
            "required data": "replication_timing, fork_speed",
            "timing data": "T(x)",
            "firing data": "-",
            "fork-speed data": fork_speed_form,
            "method": "timing_to_firing_constant_fork_speed" if implemented else "-",
            "status": "implemented" if implemented else "needs method",
        })

for target_form in V_FORMS:
    for firing_form in I_FORMS:
        rows.append({
            "target": "fork_speed",
            "target form": target_form,
            "required data": "replication_timing, firing_rate",
            "timing data": "T(x)",
            "firing data": firing_form,
            "fork-speed data": "-",
            "method": "-",
            "status": "needs method",
        })

mapping_routes = pd.DataFrame(rows)

mapping_routes

## Target: replication timing

Replication timing is the target when the goal is to predict a Repli-seq-like profile from an origin-firing programme and fork-speed assumptions. This is useful for testing whether a proposed initiation landscape is sufficient to reproduce observed timing, or for simulating how a perturbation in origin firing or fork speed would reshape timing.


### Inputs needed

To obtain timing, DNAscape needs an initiation programme and a fork-speed description. The implemented route is `I(x)` with constant fork speed `v0`, which maps to `T(x)`. The table above keeps constant, spatial, time-only, and space-time cases explicit so the remaining forward routes can be developed one by one.

### Example: firing rate to timing

Here we first infer a firing-rate profile from the example timing data, then map that firing profile forward to timing. In a real use case, the firing profile could instead come from SNS-seq, Ini-seq, ORC/MCM occupancy, accessibility, or a modelled perturbation.


In [ ]:
fit_assumptions = {
    "fork_speed": fork_speed,
    "resolution": output_resolution_kb,
    "maxiter": 6,
}

firing_fit = rmap(
    source={"replication_timing": replication_timing},
    target="firing_rate",
    assumptions=fit_assumptions,
    return_info=True,
)

firing_rate = firing_fit["values"]

timing_from_firing = rmap(
    source={"firing_rate": firing_rate},
    target="replication_timing",
    assumptions={"fork_speed": fork_speed, "resolution": output_resolution_kb},
)

In [ ]:
plotf(
    replication_timing,
    timing_from_firing,
    x_array=positions_kb,
    labels=["Timing data", "Timing from firing rate"],
    invyQ=True,
    xtitle="Chromosome position (kb)",
    ytitle="Replication timing (min)",
)

## Target: origin firing

Origin firing is the target when the goal is to infer or refine an initiation landscape from observed timing and fork-speed assumptions. This is the inverse problem most directly connected to Repli-seq-to-initiation mapping, and the output can be compared with SNS-seq, Ini-seq, OK-seq initiation zones, ORC/MCM occupancy, and chromatin features.


### Inputs needed

To obtain firing rate, DNAscape needs timing and fork speed. The implemented inverse route infers `I(x)` from `T(x)` with constant fork speed `v0`. Constant, time-only, and space-time initiation targets are kept in the table as routes to assess and implement separately.

### Example: timing to firing rate

The fitted firing rate is a spatial profile over the same genomic bins as the input timing profile.


In [ ]:
plotf(
    firing_rate,
    x_array=positions_kb,
    xtitle="Chromosome position (kb)",
    ytitle="Firing rate",
)

In [ ]:
plotf(
    replication_timing - timing_from_firing,
    x_array=positions_kb,
    labels=["Timing residual"],
    xtitle="Chromosome position (kb)",
    ytitle="Timing residual (min)",
)

## Target: fork speed

Fork speed is the target when timing and initiation are treated as known or constrained, and the goal is to infer how fast forks must move to explain the observed replication programme. This target is relevant to DNA fibre/combing, Fork-seq, polymerase progression, replication-stress perturbations, and regional fork slowing.


### Inputs needed

To obtain fork speed, DNAscape needs timing and an initiation programme. The table keeps constant, spatial, time-only, and space-time fork-speed targets separate, because each may require a different fitting strategy.

## Combined view for the runnable routes

For the routes available in this notebook, the main tracks can be aligned over the same genomic interval. This view is useful for checking whether the fitted initiation landscape, predicted timing, and observed timing are coherent.


In [ ]:
mplotf(
    [
        [replication_timing, timing_from_firing],
        firing_rate,
        replication_timing - timing_from_firing,
    ],
    mlabels=[
        ["Timing data", "Timing from firing rate"],
        ["Fitted firing rate"],
        ["Timing residual"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[True, False, False],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=[
        "Replication timing (min)",
        "Firing rate",
        "Timing residual (min)",
    ],
    figsize=(10, 2.3),
    hspace=0.18,
)